In [1]:
#google drive의 COSE362-term-project/final_dataset 폴더와 연결
from google.colab import drive

drive.mount('/content/drive')

!ls /content/drive/MyDrive/COSE362-term-project/final_dataset

import sys

sys.path.append('/content/drive/MyDrive/COSE362-term-project/final_dataset')

import os

os.chdir("/content/drive/MyDrive/COSE362-term-project/final_dataset")

Mounted at /content/drive
 africa_shape.csv	       landuse_2019.csv
 africa_shapefile	      'landuse re-preprocessing.ipynb'
 All_Countries_HIV_Rates.csv   livestock_counts.csv
 combining.ipynb	       livestock_re-preprocessing.ipynb
 crop_groupped.csv	       processed_HIV_rates.csv
 crop_indexing.ipynb	       processed_landuse_2015.csv
 disease_indexing.ipynb        processed_landuse_2019.csv
 landuse_2015.csv	       processed_livestock.csv


In [2]:
import pandas as pd
import numpy as np

In [3]:
crop_df = pd.read_csv("crop_groupped.csv")
standard_df = pd.read_csv("africa_shape.csv")
disease_df = pd.read_csv("processed_HIV_rates.csv")

In [4]:
crop_df

,country,country_code,admin_1,L1_GID,harvest_year,Cereals_Prod_Ton,Roots_tubers_Prod_Ton,Legumes_pulses_Prod_Ton,Cash_crops_Prod_Ton
0,Algeria,DZA,Adrar,L1DZA0774,1996,4.854724,11.882353,0.000000,0.000000
1,Algeria,DZA,Adrar,L1DZA0774,1998,9.552425,8.835366,1.986667,0.000000
2,Algeria,DZA,Adrar,L1DZA0774,1999,0.000000,10.380952,0.000000,0.000000
3,Algeria,DZA,Alger,L1DZA0777,1996,2.154930,14.987013,0.000000,0.000000
4,Algeria,DZA,Alger,L1DZA0777,1998,3.960932,15.728764,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
10463,Zimbabwe,ZWE,Matabeleland South,L1ZWE3661,2012,0.069491,4.776762,0.260766,0.342308
10464,Zimbabwe,ZWE,Matabeleland South,L1ZWE3661,2015,0.277564,11.173913,0.455729,0.000000
10465,Zimbabwe,ZWE,Midlands,L1ZWE3662,1994,14.144515,0.000000,0.917087,0.571800
10466,Zimbabwe,ZWE,Midlands,L1ZWE3662,2012,2.232392,18.016093,2.500376,1.655377


In [9]:
country_list = set(disease_df['country'])
crop_country_set = set(crop_df['country'])
country_list- crop_country_set

# crop data에는 congo, gabon, sao tome and principe가 없다 !
#

{'Congo', 'Gabon', 'Sao Tome and Principe'}

In [6]:
delete_country = crop_country_set - country_list
list(delete_country)

['Tanzania, United Republic of',
 'Morocco',
 'Côte dIvoire',
 'Guinea-Bissau',
 'Tanzania',
 'DRC',
 'Tunisia',
 'Uganda',
 'Republic of the Congo',
 'Madagascar',
 'South Sudan',
 'Central African Republic',
 'Botswana',
 'Egypt',
 'Algeria',
 'Mauritania',
 'Nigeria',
 'Somalia',
 'Sudan',
 'Liberia',
 'Eritrea']

In [7]:
# disease data에 없는 나라들 crop에서도 정리함
delete_condition = crop_df['country'].isin(delete_country)
crop_df = crop_df[~delete_condition]

In [10]:
# 제대로 정리됐는지 검증
crop_country_set-country_list

set()

In [11]:
check_country = ('Congo', 'Gabon', 'Sao Tome and Principe')
check_condition = disease_df['country'].isin(check_country)
check_df = disease_df[check_condition]
#len(check_df)
len(disease_df)

488

#**남은 20개의 나라에 대해서 효율적으로 지역명 매핑하기..**

1. angola 처리 이후 mismatch 쌍 147개
2. burkina faso 처리 이후 mismatch 쌍 139개
3. cameroon 처리 이후 136개
4. chad 처리 이후 126개
5. Democratic Republic of the Congo 처리 이후 109개
6. Ethiopia 처리 이후 104개
7. Gambia 처리 이후 102개
8. Ghana, Guinea 처리 이후 98개
9. Kenya 처리 이후 50개
10. Lesotho 처리 이후 46개
11. Malawi 처리 이후 15개
12. Mali 처리 이후 13개
13. Mojambique, Namibia, Niger 처리 이후 10개
14. Rwanda 처리 이후 5개
15. Senegal 처리 이후 2개
16. South Africa 처리 이후 1개
17. Togo 삭제 이후 0개

In [142]:
# 1. 'crop_df'의 (국가, 지역) 고유 키 셋 생성
# (이전에 만든 angola_mapping_dict 등이 이미 적용된 상태의 df 사용)
crop_keys = set(crop_df[['country', 'admin_1']].itertuples(index=False, name=None))

# 2. 'standard_df'의 (국가, 지역) 고유 키 셋 생성
standard_keys = set(standard_df[['ADM0_NAME', 'ADM1_NAME']].itertuples(index=False, name=None))

# 3. 'disease_df'에만 있는 "전체 문제 목록" (Master To-Do List)
all_mismatched_pairs = crop_keys - standard_keys

print(f"앞으로 수정해야 할 (국가, 지역) 쌍이 총 {len(all_mismatched_pairs)}개 남았습니다.")

앞으로 수정해야 할 (국가, 지역) 쌍이 총 0개 남았습니다.


In [143]:
# (국가, 지역) 튜플 리스트를 DataFrame으로 변환
mismatch_df = pd.DataFrame(list(all_mismatched_pairs), columns=['country', 'region_name'])

# 국가 이름으로 정렬
mismatch_df = mismatch_df.sort_values(by='country')

# 국가별로 어떤 지역 이름이 문제인지 확인
print("--- [전체 문제 목록 (국가별 정렬)] ---")
print(mismatch_df.head(50))

--- [전체 문제 목록 (국가별 정렬)] ---
Empty DataFrame
Columns: [country, region_name]
Index: []


In [144]:
len(set(mismatch_df['country']))

0

##**Angola**

#### **mapping pair**
- 'Kuando Kubango':'Cuando Cubango'
- 'Kuanza Sul':'Cuanza Sul'

#### **특이사항**

In [15]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Angola']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Angola']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Angola']['region_name'])))

angola_mapping_dict = {
    'Kuando Kubango':'Cuando Cubango',
    'Kuanza Sul':'Cuanza Sul'
}

crop_df.loc[crop_df['country'] == 'Angola', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Angola', 'admin_1'].replace(angola_mapping_dict)

print("disease data after mapping")
new_angola_df = crop_df[crop_df['country']=='Angola']
print(new_angola_df)

set(new_angola_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Angola']['ADM1_NAME'])

crop region name
['Bengo', 'Benguela', 'Bie', 'Cabinda', 'Cunene', 'Huambo', 'Huila', 'Kuando Kubango', 'Kuanza Norte', 'Kuanza Sul', 'Luanda', 'Lunda Norte', 'Lunda Sul', 'Malanje', 'Moxico', 'Namibe', 'Uige', 'Zaire']

standard region name
['Bengo', 'Benguela', 'Bie', 'Cabinda', 'Cuando Cubango', 'Cuanza Sul', 'Cunene', 'Huambo', 'Huila', 'Kuanza Norte', 'Luanda', 'Lunda Norte', 'Lunda Sul', 'Malanje', 'Moxico', 'Namibe', 'Uige', 'Zaire']



mismatch list
['Kuando Kubango', 'Kuanza Sul']
disease data after mapping
    country country_code admin_1     L1_GID  harvest_year  Cereals_Prod_Ton  \
115  Angola           AO   Bengo  L1AGO0035          1997          0.610000   
116  Angola           AO   Bengo  L1AGO0035          1998          0.700000   
117  Angola           AO   Bengo  L1AGO0035          1999          0.899956   
118  Angola           AO   Bengo  L1AGO0035          2000          1.199960   
119  Angola           AO   Bengo  L1AGO0035          2001          0.600031   
..  

set()

In [16]:
until_angola_df = crop_df

In [17]:
set(disease_df[disease_df['country']=='Angola']['year'])

{2015}

In [18]:
set(crop_df[crop_df['country']=='Angola']['harvest_year'])

{1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017}

##**Burkina Faso**

#### **mapping pair**
- 'Boucle du Mouhoun':'Boucle Du Mouhoun',
- 'Centre-Est': 'Centre-est',
- 'Centre-Nord':'Centre-nord',
- 'Centre-Ouest':'Centre-ouest',
- 'Centre-Sud':'Centre-sud',
- 'Haut-Bassins':'Hauts-bassins',
- 'Plateau-Central':'Plateau Central',
- 'Sud-Ouest':'Sud-ouest'

#### **특이사항**


In [19]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Burkina Faso']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Burkina Faso']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Burkina Faso']['region_name'])))

burkina_mapping_dict = {
    'Boucle du Mouhoun':'Boucle Du Mouhoun',
    'Centre-Est': 'Centre-est',
    'Centre-Nord':'Centre-nord',
    'Centre-Ouest':'Centre-ouest',
    'Centre-Sud':'Centre-sud',
    'Haut-Bassins':'Hauts-bassins',
    'Plateau-Central':'Plateau Central',
    'Sud-Ouest':'Sud-ouest'
}

crop_df.loc[crop_df['country'] == 'Burkina Faso', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Burkina Faso', 'admin_1'].replace(burkina_mapping_dict)

print("disease data after mapping")
new_burkina_df = crop_df[crop_df['country']=='Burkina Faso']
print(new_burkina_df)

set(new_burkina_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Burkina Faso']['ADM1_NAME'])

crop region name
['Boucle du Mouhoun', 'Cascades', 'Centre', 'Centre-Est', 'Centre-Nord', 'Centre-Ouest', 'Centre-Sud', 'Est', 'Haut-Bassins', 'Nord', 'Plateau-Central', 'Sahel', 'Sud-Ouest']

standard region name
['Boucle Du Mouhoun', 'Cascades', 'Centre', 'Centre-est', 'Centre-nord', 'Centre-ouest', 'Centre-sud', 'Est', 'Hauts-bassins', 'Nord', 'Plateau Central', 'Sahel', 'Sud-ouest']



mismatch list
['Boucle du Mouhoun', 'Centre-Est', 'Centre-Nord', 'Centre-Ouest', 'Centre-Sud', 'Haut-Bassins', 'Plateau-Central', 'Sud-Ouest']
disease data after mapping
          country country_code            admin_1     L1_GID  harvest_year  \
499  Burkina Faso          BFA  Boucle Du Mouhoun  L1BFA0216          1995   
500  Burkina Faso          BFA  Boucle Du Mouhoun  L1BFA0216          1996   
501  Burkina Faso          BFA  Boucle Du Mouhoun  L1BFA0216          1997   
502  Burkina Faso          BFA  Boucle Du Mouhoun  L1BFA0216          1998   
503  Burkina Faso          BFA  Boucle Du Mouho

set()

In [20]:
until_burkina_df = crop_df

In [21]:
set(disease_df[disease_df['country']=='Burkina Faso']['year'])

{2003, 2010}

In [22]:
set(crop_df[crop_df['country']=='Burkina Faso']['harvest_year'])

{1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018}

##**Cameroon**

#### **mapping pair**
- 'Extrême-Nord':'Extrême - Nord'
- 'Nord-Ouest':'Nord - Ouest'
- 'Sud-Ouest':'Sud - Ouest'

#### **특이사항**
- disease data가 {2004, 2011, 2018}에 대하여 존재
- crop data는 {1989, 2001, 2009, 2010, 2011}에 대해서 존재
- 두 데이터가 겹치는 연도는 2011년 뿐

In [23]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Cameroon']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Cameroon']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Cameroon']['region_name'])))

cameroon_mapping_dict = {
    'Extrême-Nord':'Extrême - Nord',
    'Nord-Ouest':'Nord - Ouest',
    'Sud-Ouest':'Sud - Ouest'
}

crop_df.loc[crop_df['country'] == 'Cameroon', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Cameroon', 'admin_1'].replace(cameroon_mapping_dict)

print("disease data after mapping")
new_cameroon_df = crop_df[crop_df['country']=='Cameroon']
print(new_cameroon_df)

set(new_cameroon_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Cameroon']['ADM1_NAME'])

crop region name
['Adamaoua', 'Centre', 'Est', 'Extrême-Nord', 'Littoral', 'Nord', 'Nord-Ouest', 'Ouest', 'Sud', 'Sud-Ouest']

standard region name
['Adamaoua', 'Centre', 'Est', 'Extrême - Nord', 'Littoral', 'Nord', 'Nord - Ouest', 'Ouest', 'Sud', 'Sud - Ouest']



mismatch list
['Extrême-Nord', 'Nord-Ouest', 'Sud-Ouest']
disease data after mapping
      country country_code         admin_1     L1_GID  harvest_year  \
891  Cameroon          CMR          Centre  L1CMR0541          1989   
892  Cameroon          CMR          Centre  L1CMR0541          2001   
893  Cameroon          CMR          Centre  L1CMR0541          2009   
894  Cameroon          CMR          Centre  L1CMR0541          2010   
895  Cameroon          CMR          Centre  L1CMR0541          2011   
896  Cameroon          CMR             Est  L1CMR0542          1989   
897  Cameroon          CMR             Est  L1CMR0542          2001   
898  Cameroon          CMR             Est  L1CMR0542          2009   
899  Camer

set()

In [24]:
set(disease_df[disease_df['country']=='Cameroon']['year'])

{2004, 2011, 2018}

In [25]:
set(crop_df[crop_df['country']=='Cameroon']['harvest_year'])

{1989, 2001, 2009, 2010, 2011}


##**Chad**

#### **mapping pair**


#### **특이사항**
- 복잡함.. disease 매핑 case 따라함

In [26]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Chad']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Chad']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Chad']['region_name'])))

crop region name
['Barh el Gazel', 'Batha', 'Chari-Baguirmi', 'Guera', 'Hadjer-Lamis', 'Kanem', 'Lac', 'Logone Occidental', 'Logone Oriental', 'Mandoul', 'Mayo-Kebbi Est', 'Mayo-Kebbi Ouest', 'Moyen-Chari', 'Ouaddai', 'Ouaddaï', 'Salamat', 'Sila', 'Tandjile', 'Wadi Fira']

standard region name
['Assongha', 'Baguirmi', 'Barh Koh', 'Barl El Gazal', 'Batha Est', 'Batha Ouest', 'Biltine', 'Borkou', 'Daraba', 'Ennedi', 'Guera', 'Hadjer Lamis', 'Kabia', 'Kanem', 'Lac', 'Lac Iro', 'Logone Occidental', 'Logone Oriental', 'Mandoul', 'Mayo-Boneye', 'Mayo-Dala', 'Mont De Lam', 'Ouaddai', 'Salamat', 'Sila', 'Tandjile Est', 'Tandjile Ouest', 'Tibesti']



mismatch list
['Barh el Gazel', 'Batha', 'Chari-Baguirmi', 'Hadjer-Lamis', 'Mayo-Kebbi Est', 'Mayo-Kebbi Ouest', 'Moyen-Chari', 'Ouaddaï', 'Tandjile', 'Wadi Fira']


In [27]:
#매핑할 차드(Chad) 지역 딕셔너리
mapping_dict_chad = {
    # 오타 (l vs h)
    'Barh el Gazel': 'Barl El Gazal',

    # 악센트 제거
    'Guéra': 'Guera',

    # 하이픈 제거
    'Hadjer-Lamis': 'Hadjer Lamis',

    # 악센트 제거
    'Ouaddaï': 'Ouaddai',

    # 역사적 이름 변경 (Wadi Fira(신) -> Biltine(구))
    'Wadi Fira': 'Biltine'
}

# 삭제할 차드(Chad) 지역 목록
regions_to_delete_chad = [
    # (1:N 문제) standard는 'Batha Est', 'Batha Ouest'로 2개임
    'Batha',

    # (N:1 문제) standard는 'Borkou', 'Tibesti'로 2개임
    'Borkou/Tibesti',

    # (L1:L2 문제) standard는 하위 L2(Baguirmi)만 포함
    'Chari-Baguirmi',

    # (L1:L2 문제) standard는 하위 L2들만 포함
    'Mayo-Kebbi Est',
    'Mayo-Kebbi Ouest',
    'Moyen-Chari',

    # (수도 누락) standard 인덱스에 수도가 없음
    "N'Djaména",

    # (1:N 문제) standard는 'Tandjile Est', 'Tandjile Ouest'로 2개임
    'Tandjile'
]

In [28]:
print(f"--- 차드(Chad) 데이터 정리 시작 ---")

try:
    # (검증) 원본 데이터 행 수
    original_count = len(crop_df[crop_df['country'] == 'Chad'])
    print(f"원본 'Chad' 행 개수: {original_count}")

    # 3-1. 1단계: 매핑 불가능한 행 우선 삭제

    # 삭제할 행의 인덱스를 찾습니다.
    indices_to_drop = crop_df[
        (crop_df['country'] == 'Chad') &
        (crop_df['admin_1'].isin(regions_to_delete_chad))
    ].index

    # 원본(crop_df)에서 해당 인덱스를 삭제합니다.
    crop_df_cleaned = crop_df.drop(indices_to_drop)

    print(f"1단계: 매핑 불가 지역 {len(indices_to_drop)}개 행 삭제 완료.")

    # 3-2. 2단계: 오타 및 역사적 이름 매핑

    # (중요) 'Chad' 국가에만 .loc[]를 사용해 안전하게 적용합니다.
    mask = (crop_df_cleaned['country'] == 'Chad')
    crop_df_cleaned.loc[mask, 'admin_1'] = crop_df_cleaned.loc[mask, 'admin_1'].replace(mapping_dict_chad)

    print("2단계: 오타 및 역사적 이름 매핑 완료.")

    # 3-3. (중요) 원본 DataFrame 변수 덮어쓰기
    crop_df = crop_df_cleaned

    print(f"최종 'Chad' 행 개수: {len(crop_df[crop_df['country'] == 'Chad'])}")
    print("--- 차드(Chad) 데이터 정리 완료 ---")

except NameError:
    print("오류: 'disease_df'가 메모리에 없습니다. 먼저 로드해주세요.")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")

--- 차드(Chad) 데이터 정리 시작 ---
원본 'Chad' 행 개수: 589
1단계: 매핑 불가 지역 205개 행 삭제 완료.
2단계: 오타 및 역사적 이름 매핑 완료.
최종 'Chad' 행 개수: 384
--- 차드(Chad) 데이터 정리 완료 ---


In [ ]:
chad_disease_region_name = set(['Barh El Gazal', 'Batha', 'Borkou/Tibesti', 'Chari Baguirmi', 'Ennedi', 'Guéra', 'Hadjer-Lamis', 'Kanem', 'Lac', 'Logone Occidental', 'Logone Oriental', 'Mandoul', 'Mayo Kebbi Est', 'Mayo Kebbi Ouest', 'Moyen Chari', "N'Djaména", 'Ouaddaï', 'Salamat', 'Sila', 'Tandjilé', 'Wadi Fira'])
chad_crop_region_name=set(['Barh el Gazel', 'Batha', 'Chari-Baguirmi', 'Guera', 'Hadjer-Lamis', 'Kanem', 'Lac', 'Logone Occidental', 'Logone Oriental', 'Mandoul', 'Mayo-Kebbi Est', 'Mayo-Kebbi Ouest', 'Moyen-Chari', 'Ouaddai', 'Ouaddaï', 'Salamat', 'Sila', 'Tandjile', 'Wadi Fira'])

chad_disease_region_name-chad_crop_region_name

{'Barh El Gazal',
 'Borkou/Tibesti',
 'Chari Baguirmi',
 'Ennedi',
 'Guéra',
 'Mayo Kebbi Est',
 'Mayo Kebbi Ouest',
 'Moyen Chari',
 "N'Djaména",
 'Tandjilé'}

In [29]:
until_chad_df = crop_df

In [ ]:
set(crop_df[(crop_df['country']=='Chad')]['harvest_year'])

{1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1998,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017}

In [ ]:
set(disease_df[disease_df['country']=='Chad']['year'])

{2014}

##**Democratic Republic of the Congo**

#### **mapping pair**


#### **특이사항**
- crop 데이터엔 1994,1995,1996
- disease 데이터엔 2007,2013,2023 존재..
- 이렇게 되면 drc disease data는 crop data가 없음..
- crop 에서 drc 데이터 다 지움

In [32]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Democratic Republic of the Congo']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Democratic Republic of the Congo']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Democratic Republic of the Congo']['region_name'])))

crop region name
['Bas-Uele', 'Haut-Lomami', 'Haut-Uele', 'Ituri', 'Kasaï', 'Kinshasa', 'Kwango', 'Kwilu', 'Lualaba', 'Mai-Ndombe', 'Maniema', 'Mongala', 'Nord-Kivu', 'Nord-Ubangi', 'Sankuru', 'Sud-Kivu', 'Sud-Ubangi', 'Tanganyika', 'Tshopo', 'Tshuapa', 'Équateur']

standard region name
['Bandundu', 'Bas-Congo', 'Equateur', 'Kasai Occidental', 'Kasai Oriental', 'Katanga', 'Kinshasa', 'Maniema', 'Nord-Kivu', 'Orientale', 'Sud-Kivu']



mismatch list
['Bas-Uele', 'Haut-Lomami', 'Haut-Uele', 'Ituri', 'Kasaï', 'Kwango', 'Kwilu', 'Lualaba', 'Mai-Ndombe', 'Mongala', 'Nord-Ubangi', 'Sankuru', 'Sud-Ubangi', 'Tanganyika', 'Tshopo', 'Tshuapa', 'Équateur']


In [ ]:
set(crop_df[(crop_df['country']=='Democratic Republic of the Congo')]['harvest_year'])

{1994, 1995, 1996}

In [35]:
disease_df[disease_df['country']=='Democratic Republic of the Congo']

,year,country,region_name,infection_rate
77,2007,Democratic Republic of the Congo,Bandundu,0.525210
78,2007,Democratic Republic of the Congo,Bas-Congo,1.376147
79,2007,Democratic Republic of the Congo,Equateur,1.179245
80,2007,Democratic Republic of the Congo,Kasai Occidental,0.718391
81,2007,Democratic Republic of the Congo,Kasai Oriental,1.593137
82,2007,Democratic Republic of the Congo,Katanga,1.985981
83,2007,Democratic Republic of the Congo,Kinshasa,1.876877
84,2007,Democratic Republic of the Congo,Maniema,0.462428
85,2007,Democratic Republic of the Congo,Nord-Kivu,1.335113
86,2007,Democratic Republic of the Congo,Orientale,2.264151


In [34]:
crop_df = crop_df[~(crop_df['country']=='Democratic Republic of the Congo')]

In [36]:
until_drc_df = crop_df

In [ ]:
crop_df = until_drc_df


##**Ethiopia**

#### **mapping pair**
- 'Addis Abeba':'Addis Ababa',
- 'Benshangul-Gumaz':'Beneshangul Gumu',
- 'Gambela Peoples':'Gambela',
- 'Harari People':'Hareri',
- 'Southern Nations, Nationalities':'SNNPR'


#### **특이사항**
{1991, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018} 에 대한 crop data만 존재해서 disease data {2005, 2011, 2016} 에서 2005년에 대한 데이터 부족

In [39]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Ethiopia']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Ethiopia']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Ethiopia']['region_name'])))


ethiopia_mapping_dict = {
    'Addis Abeba':'Addis Ababa',
    'Benshangul-Gumaz':'Beneshangul Gumu',
    'Gambela Peoples':'Gambela',
    'Harari People':'Hareri',
    'Southern Nations, Nationalities':'SNNPR'
}

crop_df.loc[crop_df['country'] == 'Ethiopia', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Ethiopia', 'admin_1'].replace(ethiopia_mapping_dict)

print("disease data after mapping")
new_ethiopia_df = crop_df[crop_df['country']=='Ethiopia']
print(new_ethiopia_df)

set(new_ethiopia_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Ethiopia']['ADM1_NAME'])

crop region name
['Addis Abeba', 'Afar', 'Amhara', 'Benshangul-Gumaz', 'Dire Dawa', 'Gambela Peoples', 'Harari People', 'Oromia', 'Somali', 'Southern Nations, Nationalities', 'Tigray']

standard region name
['Addis Ababa', 'Afar', 'Amhara', 'Beneshangul Gumu', 'Dire Dawa', 'Gambela', 'Hareri', 'Oromia', 'SNNPR', 'Somali', 'Tigray']



mismatch list
['Addis Abeba', 'Benshangul-Gumaz', 'Gambela Peoples', 'Harari People', 'Southern Nations, Nationalities']
disease data after mapping
       country country_code      admin_1     L1_GID  harvest_year  \
1784  Ethiopia          ETH  Addis Ababa  L1ETH0917          1991   
1785  Ethiopia          ETH         Afar  L1ETH0918          1991   
1786  Ethiopia          ETH         Afar  L1ETH0918          2007   
1787  Ethiopia          ETH         Afar  L1ETH0918          2008   
1788  Ethiopia          ETH         Afar  L1ETH0918          2009   
...        ...          ...          ...        ...           ...   
1907  Ethiopia          ETH     

set()

In [40]:
set(crop_df[(crop_df['country']=='Ethiopia')]['harvest_year'])

{1991, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018}

In [41]:
set(disease_df[(disease_df['country']=='Ethiopia')]['year'])

{2005, 2011, 2016}

##**Gambia**

#### **mapping pair**


#### **특이사항**
disease : {2013}
crop : {2000, 1994, 1998}
crop 데이터에서 gambia 삭제함

In [42]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Gambia']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Gambia']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Gambia']['region_name'])))

crop region name
['Lower River', 'Maccarthy Island', 'North Bank', 'Upper River', 'Western']

standard region name
['Central River', 'Kanifing Municipal Council', 'Lower River', 'North Bank', 'Upper River', 'West Coast']



mismatch list
['Maccarthy Island', 'Western']


In [43]:
print(set(disease_df[(disease_df['country']=='Gambia')]['year']))
print(set(crop_df[(crop_df['country']=='Gambia')]['harvest_year']))

{2013}
{2000, 1994, 1998}


In [44]:
crop_df = crop_df[~(crop_df['country']=='Gambia')]

In [45]:
until_gambia_df = crop_df

##**Ghana, Guinea**

#### **mapping pair**


#### **특이사항**
- 가나
  - disease : {2003, 2014}
  - crop : {1997, 1998, 1999, 2000, 2001}
  - crop에서 전체 삭제..

- 기니
  - disease : {2018, 2012, 2005}
  - crop : {2016, 2017, 1998, 2010, 2011, 2012, 2013, 2014, 2015}
  - disease의 2005, 2018년도는 crop 결측치

In [48]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Guinea']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Guinea']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Guinea']['region_name'])))

guinea_mapping_dict = {
    'Boké':'Boke',
    'Labé':'Labe',
    'Nzérékoré':'Nzerekore'
}

crop_df.loc[crop_df['country'] == 'Guinea', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Guinea', 'admin_1'].replace(guinea_mapping_dict)

print("disease data after mapping")
new_guinea_df = crop_df[crop_df['country']=='Guinea']
print(new_guinea_df)

set(new_guinea_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Guinea']['ADM1_NAME'])

crop region name
['Boké', 'Conakry', 'Faranah', 'Kankan', 'Kindia', 'Labé', 'Mamou', 'Nzérékoré']

standard region name
['Boke', 'Conakry', 'Faranah', 'Kankan', 'Kindia', 'Labe', 'Mamou', 'Nzerekore']



mismatch list
['Boké', 'Labé', 'Nzérékoré']
disease data after mapping
     country country_code    admin_1     L1_GID  harvest_year  \
1973  Guinea          GIN       Boke  L1GIN1015          2010   
1974  Guinea          GIN       Boke  L1GIN1015          2011   
1975  Guinea          GIN       Boke  L1GIN1015          2012   
1976  Guinea          GIN       Boke  L1GIN1015          2013   
1977  Guinea          GIN       Boke  L1GIN1015          2014   
...      ...          ...        ...        ...           ...   
2034  Guinea          GIN  Nzerekore  L1GIN1022          2013   
2035  Guinea          GIN  Nzerekore  L1GIN1022          2014   
2036  Guinea          GIN  Nzerekore  L1GIN1022          2015   
2037  Guinea          GIN  Nzerekore  L1GIN1022          2016   
2038  Guin

set()

In [ ]:
print(set(disease_df[(disease_df['country']=='Ghana')]['year']))
print(set(crop_df[(crop_df['country']=='Ghana')]['harvest_year']))

{2003, 2014}
{1997, 1998, 1999, 2000, 2001}


In [49]:
crop_df = crop_df[~(crop_df['country']=='Ghana')]

In [ ]:
print(set(disease_df[(disease_df['country']=='Guinea')]['year']))
print(set(crop_df[(crop_df['country']=='Guinea')]['harvest_year']))

{2018, 2012, 2005}
{2016, 2017, 1998, 2010, 2011, 2012, 2013, 2014, 2015}


In [50]:
until_guinea_df = crop_df

In [74]:
crop_df = until_guinea_df

##**Kenya**

#### **mapping pair**
crop은 카운티 단위, standard는 주 단위여서 crop의 카운티 당 생산량을 합계해서 주 당 생산량을 집계하도록 했다.



#### **특이사항**
- crop 데이터에서 disease에 있는 연도인 2003, 2008년의 데이터만 남김

In [77]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Kenya']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Kenya']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Kenya']['region_name'])))

crop region name
['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo-Marakwet', 'Embu', 'Garissa', 'Homa Bay', 'Isiolo', 'Kajiado', 'Kakamega', 'Kericho', 'Kiambu', 'Kilifi', 'Kirinyaga', 'Kisii', 'Kisumu', 'Kitui', 'Kwale', 'Laikipia', 'Lamu', 'Machakos', 'Makueni', 'Mandera', 'Marsabit', 'Meru', 'Migori', 'Mombasa', "Murang'a", 'Muranga', 'Nairobi', 'Nakuru', 'Nandi', 'Narok', 'Nyamira', 'Nyandarua', 'Nyeri', 'Samburu', 'Siaya', 'Taita Taveta', 'Tana River', 'Tharaka Nithi', 'Tharaka-Nithi', 'Trans Nzoia', 'Turkana', 'Uasin Gishu', 'Vihiga', 'Wajir', 'West Pokot']

standard region name
['Central', 'Coast', 'Eastern', 'Nairobi', 'North Eastern', 'Nyanza', 'Rift Valley', 'Western']



mismatch list
['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo-Marakwet', 'Embu', 'Garissa', 'Homa Bay', 'Isiolo', 'Kajiado', 'Kakamega', 'Kericho', 'Kiambu', 'Kilifi', 'Kirinyaga', 'Kisii', 'Kisumu', 'Kitui', 'Kwale', 'Laikipia', 'Lamu', 'Machakos', 'Makueni', 'Mandera', 'Marsabit', 'Meru', 'Migori', 'Momba

In [78]:
print(set(disease_df[(disease_df['country']=='Kenya')]['year']))
print(set(crop_df[(crop_df['country']=='Kenya')]['harvest_year']))

{2008, 2003}
{1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2019}


In [79]:
# 1. 케냐 데이터 중 유지할 연도를 리스트로 정의
years_to_keep = [2008, 2003]

# 2. 'Kenya'가 아닌 행을 선택하는 조건 (다른 나라 데이터는 모두 보존)
cond_non_kenya = (crop_df['country'] != 'Kenya')

# 3. 'Kenya'이면서, 'harvest_year'가 [2008, 2003] 리스트에 포함되는 행을 선택하는 조건
cond_kenya_ok_years = (crop_df['country'] == 'Kenya') & crop_df['harvest_year'].isin(years_to_keep)

# 4. 두 조건 중 하나라도 True인 모든 행을 선택하여 crop_df를 덮어씀
# (즉, 케냐가 아니거나, 케냐이면서 연도가 맞는 경우)
crop_df = crop_df[cond_non_kenya | cond_kenya_ok_years]

# (확인) 케냐의 harvest_year 확인
print(set(crop_df[crop_df['country'] == 'Kenya']['harvest_year']))

{2008, 2003}


In [80]:
print("--- 케냐(Kenya) 데이터 표준화(Many-to-One) 시작 ---")

# --- 1. 47개 카운티 -> 8개 주 매핑 사전 정의 ---
# (사용자의 crop region name 리스트에 있는 철자 변형 포함)
county_to_province_map = {
    # Central (5)
    'Kiambu': 'Central',
    'Kirinyaga': 'Central',
    "Murang'a": 'Central',
    'Muranga': 'Central',  # 사용자의 철자 변형
    'Nyandarua': 'Central',
    'Nyeri': 'Central',

    # Coast (6)
    'Kilifi': 'Coast',
    'Kwale': 'Coast',
    'Lamu': 'Coast',
    'Mombasa': 'Coast',
    'Taita Taveta': 'Coast',
    'Tana River': 'Coast',

    # 'Eastern' (8)
    'Embu': 'Eastern',
    'Isiolo': 'Eastern',
    'Kitui': 'Eastern',
    'Machakos': 'Eastern',
    'Makueni': 'Eastern',
    'Marsabit': 'Eastern',
    'Meru': 'Eastern',
    'Tharaka Nithi': 'Eastern',
    'Tharaka-Nithi': 'Eastern', # 사용자의 철자 변형

    # Nairobi (1)
    'Nairobi': 'Nairobi',

    # North Eastern (3)
    'Garissa': 'North Eastern',
    'Mandera': 'North Eastern',
    'Wajir': 'North Eastern',

    # Nyanza (6)
    'Homa Bay': 'Nyanza',
    'Kisii': 'Nyanza',
    'Kisumu': 'Nyanza',
    'Migori': 'Nyanza',
    'Nyamira': 'Nyanza',
    'Siaya': 'Nyanza',

    # Rift Valley (14)
    'Baringo': 'Rift Valley',
    'Bomet': 'Rift Valley',
    'Elgeyo-Marakwet': 'Rift Valley',
    'Kajiado': 'Rift Valley',
    'Kericho': 'Rift Valley',
    'Laikipia': 'Rift Valley',
    'Nakuru': 'Rift Valley',
    'Nandi': 'Rift Valley',
    'Narok': 'Rift Valley',
    'Samburu': 'Rift Valley',
    'Trans Nzoia': 'Rift Valley',
    'Turkana': 'Rift Valley',
    'Uasin Gishu': 'Rift Valley',
    'West Pokot': 'Rift Valley',

    # Western (4)
    'Bungoma': 'Western',
    'Busia': 'Western',
    'Kakamega': 'Western',
    'Vihiga': 'Western'
}

size_map = {
    'Central': {
        'Kiambu': 2543.4,
        'Kirinyaga': 1479.1,
        "Murang'a": 2558.8,  # Muranga와 동일
        'Muranga': 2558.8,   # Murang'a와 동일
        'Nyandarua': 3304.3,
        'Nyeri': 3337.1,
    },
    'Coast': {
        'Kilifi': 12609.8,
        'Kwale': 8270.3,
        'Lamu': 6273.1,
        'Mombasa': 219.9,
        'Taita Taveta': 17083.9,
        'Tana River': 38436.8,
    },
    'Eastern': {
        'Embu': 2820.7,
        'Isiolo': 25336.1,
        'Kitui': 30429.5,
        'Machakos': 6208.2,
        'Makueni': 8169.8,
        'Marsabit': 70961.3,
        'Meru': 7003.1,
        'Tharaka Nithi': 2609.5,  # Tharaka-Nithi와 동일
        'Tharaka-Nithi': 2609.5, # Tharaka Nithi와 동일
    },
    'Nairobi': {
        'Nairobi': 694.9, # 나이로비는 카운티이자 도시
    },
    'North Eastern': {
        'Garissa': 44753.0,
        'Mandera': 25939.8,
        'Wajir': 56685.8,
    },
    'Nyanza': {
        'Homa Bay': 3154.7,
        'Kisii': 1317.9,
        'Kisumu': 2085.9,
        'Migori': 2586.4,
        'Nyamira': 912.5,
        'Siaya': 2530.4,
    },
    'Rift Valley': {
        'Baringo': 11075.3,
        'Bomet': 2037.4,
        'Elgeyo-Marakwet': 3049.7,
        'Kajiado': 21292.7,
        'Kericho': 2454.5,
        'Laikipia': 9462.0,
        'Nakuru': 7496.5,
        'Nandi': 2884.4,
        'Narok': 17921.2,
        'Samburu': 21022.2,
        'Trans Nzoia': 2495.5,
        'Turkana': 68680.3, # 가장 면적이 넓은 카운티 중 하나
        'Uasin Gishu': 3345.2,
        'West Pokot': 9169.4,
    },
    'Western': {
        'Bungoma': 2069.1,
        'Busia': 1694.5,
        'Kakamega': 3033.8,
        'Vihiga': 531.3, # 면적이 가장 작은 카운티 중 하나
    }
}

flat_size_map = {
    county: size
    for region_counties in size_map.values()
    for county, size in region_counties.items()
}

--- 케냐(Kenya) 데이터 표준화(Many-to-One) 시작 ---


In [81]:

# (오류 방지)
try:
    # --- 2. 케냐 데이터 분리 ---
    # 케냐가 아닌 다른 나라 데이터는 그대로 보존
    df_others = crop_df[crop_df['country'] != 'Kenya'].copy()

    # 케냐 데이터만 추출
    df_kenya = crop_df[crop_df['country'] == 'Kenya'].copy()

    if df_kenya.empty:
        print("경고: crop_df에 'Kenya' 데이터가 없습니다. 작업을 건너뜁니다.")
    else:
        # 'admin_1'을 매핑 사전을 이용해 'standard_region'으로변환
        df_kenya['size'] = df_kenya['admin_1'].map(flat_size_map)
        df_kenya['size'] = df_kenya['size'].fillna(0)
        df_kenya['standard_region'] = df_kenya['admin_1'].map(county_to_province_map)

        unmapped = df_kenya[df_kenya['standard_region'].isnull()]['admin_1'].unique()
        if len(unmapped) > 0:
            print(f"경고: 매핑에 실패한 카운티가 있습니다 -> {unmapped}")

        # [수정됨] 집계할 컬럼 리스트
        agg_columns = ['Cereals_Prod_Ton', 'Roots_tubers_Prod_Ton', 'Legumes_pulses_Prod_Ton', 'Cash_crops_Prod_Ton']

        # --- 3. 'total_production' (분자) 계산 ---
        # 가중 평균의 분자 (yield * size)를 agg_columns의 모든 열에 대해 계산합니다.
        total_prod_cols = []
        for col in agg_columns:
            prod_col_name = f'total_prod_{col}'
            df_kenya[prod_col_name] = df_kenya[col] * df_kenya['size']
            total_prod_cols.append(prod_col_name)




        # --- 4. Groupby 및 합산 ---
        # 이제 'total_production' 열들과 'size' (분모)를 그룹별로 합산합니다.
        group_keys = ['country','country_code', 'harvest_year', 'standard_region']

        # 합산할 열 목록: ['total_prod_crop_yield_A', 'total_prod_crop_yield_B', 'size']
        cols_to_sum = total_prod_cols + ['size']

        df_kenya_agg = df_kenya.groupby(group_keys)[cols_to_sum].sum().reset_index()

        # --- 5. 최종 가중 평균 계산 ---
        # (total_prod / total_size)
        for col in agg_columns:
            prod_col_name = f'total_prod_{col}'

            # 분모(df_kenya_agg['size'])가 0일 경우 0으로 나누기 오류 방지
            # np.where(조건, 참일 때 값, 거짓일 때 값)
            df_kenya_agg[col] = np.where(
                df_kenya_agg['size'] == 0,
                np.nan,  # 면적이 0이면 결과는 NaN
                df_kenya_agg[prod_col_name] / df_kenya_agg['size']
            )

            # 임시로 사용했던 'total_prod_' 열 삭제
            df_kenya_agg = df_kenya_agg.drop(columns=[prod_col_name])


        df_kenya_agg = df_kenya_agg.drop(columns=['size'])

        print(f"1단계: 'Kenya' 데이터 {len(df_kenya)}행을 {len(df_kenya_agg)}행으로 집계 (sum 사용).")

        df_kenya_agg = df_kenya_agg.rename(columns={'standard_region': 'admin_1'})
        df_kenya_agg['L1_GID'] = np.nan

        # --- 4. 다른 국가 데이터와 다시 합치기 ---
        crop_df = pd.concat([df_others, df_kenya_agg], ignore_index=True)

        print(f"--- [최종 작업 완료] 케냐(Kenya) 데이터 정리 완료 ---")

        # (검증)
        print("\n--- 검증 (케냐 8개 주) ---")
        standard_kenya_set = set(['Central', 'Coast', 'Eastern', 'Nairobi', 'North Eastern', 'Nyanza', 'Rift Valley', 'Western'])
        disease_set_kenya = set(crop_df[crop_df['country'] == 'Kenya']['admin_1'])
        print(f"검증 결과 (set()가 나와야 함): {disease_set_kenya - standard_kenya_set}")

except NameError:
    print("오류: 'crop_df'가 메모리에 없습니다. 먼저 로드해주세요.")
except KeyError as e:
    print(f"오류: DataFrame에 '{e.key}' 컬럼이 없습니다. (혹시 'infection_rate' 컬럼명이 다른가요?)")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")

1단계: 'Kenya' 데이터 92행을 14행으로 집계 (sum 사용).
--- [최종 작업 완료] 케냐(Kenya) 데이터 정리 완료 ---

--- 검증 (케냐 8개 주) ---
검증 결과 (set()가 나와야 함): set()


In [73]:
crop_df.columns

Index(['country', 'country_code', 'admin_1', 'L1_GID', 'harvest_year',
       'Cereals_Prod_Ton', 'Roots_tubers_Prod_Ton', 'Legumes_pulses_Prod_Ton',
       'Cash_crops_Prod_Ton'],
      dtype='object')

In [84]:
until_kenya_df = crop_df

##**Lesotho**

#### **mapping pair**
- 'Butha-Buthe':'Butha Buthe'
- 'Mohales Hoek':"Mohale's Hoek"
- 'Qachas Nek':"Qacha's Nek"
- 'Thaba-Tseka':'Thaba Tseka'

#### **특이사항**

In [85]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Lesotho']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Lesotho']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Lesotho']['region_name'])))

crop region name
['Berea', 'Butha-Buthe', 'Leribe', 'Mafeteng', 'Maseru', "Mohale's Hoek", 'Mohales Hoek', 'Mokhotlong', "Qacha's Nek", 'Qachas Nek', 'Quthing', 'Thaba-Tseka']

standard region name
['Berea', 'Butha Buthe', 'Leribe', 'Mafeteng', 'Maseru', "Mohale's Hoek", 'Mokhotlong', "Qacha's Nek", 'Quthing', 'Thaba Tseka']



mismatch list
['Butha-Buthe', 'Mohales Hoek', 'Qachas Nek', 'Thaba-Tseka']


In [86]:
print(set(disease_df[(disease_df['country']=='Lesotho')]['year']))
print(set(crop_df[(crop_df['country']=='Lesotho')]['harvest_year']))

{2009, 2004, 2014}
{1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022}


In [87]:
lesotho_mapping_dict = {
    'Butha-Buthe':'Butha Buthe',
    'Mohales Hoek':"Mohale's Hoek",
    'Qachas Nek':"Qacha's Nek",
    'Thaba-Tseka':'Thaba Tseka'
}

crop_df.loc[crop_df['country'] == 'Lesotho', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Lesotho', 'admin_1'].replace(lesotho_mapping_dict)

print("disease data after mapping")
new_lesotho_df = crop_df[crop_df['country']=='Lesotho']
print(new_lesotho_df)

set(new_lesotho_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Lesotho']['ADM1_NAME'])

disease data after mapping
      country country_code        admin_1     L1_GID  harvest_year  \
1398  Lesotho           LS          Berea  L1LSO1721          1982   
1399  Lesotho           LS          Berea  L1LSO1721          1983   
1400  Lesotho           LS          Berea  L1LSO1721          1984   
1401  Lesotho           LS          Berea  L1LSO1721          1985   
1402  Lesotho           LS          Berea  L1LSO1721          1986   
...       ...          ...            ...        ...           ...   
1794  Lesotho          LSO    Thaba Tseka  L1LSO1730          1997   
1795  Lesotho          LSO  Mohale's Hoek  L1LSO1726          1990   
1796  Lesotho          LSO  Mohale's Hoek  L1LSO1726          1997   
1797  Lesotho          LSO    Qacha's Nek  L1LSO1728          1990   
1798  Lesotho          LSO    Qacha's Nek  L1LSO1728          1994   

      Cereals_Prod_Ton  Roots_tubers_Prod_Ton  Legumes_pulses_Prod_Ton  \
1398          2.226357                    0.0             

set()

In [88]:
until_lesotho_df=crop_df

##**Malawi**

#### **mapping pair**

#### **특이사항**

In [67]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Malawi']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Malawi']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Malawi']['region_name'])))

crop region name
['Balaka', 'Blantyre', 'Central', 'Chikwawa', 'Chiradzulu', 'Chitipa', 'Dedza', 'Dowa', 'Karonga', 'Kasungu', 'Likoma', 'Lilongwe', 'Machinga', 'Mangochi', 'Mchinji', 'Mulanje', 'Mwanza', 'Mzimba', 'Neno', 'Nkhata Bay', 'Nkhotakota', 'Northern', 'Nsanje', 'Ntcheu', 'Ntchisi', 'Phalombe', 'Rumphi', 'Salima', 'Southern', 'Thyolo', 'Zomba']

standard region name
['Area under National Administration', 'Central Region', 'Northern Region', 'Southern Region']



mismatch list
['Balaka', 'Blantyre', 'Central', 'Chikwawa', 'Chiradzulu', 'Chitipa', 'Dedza', 'Dowa', 'Karonga', 'Kasungu', 'Likoma', 'Lilongwe', 'Machinga', 'Mangochi', 'Mchinji', 'Mulanje', 'Mwanza', 'Mzimba', 'Neno', 'Nkhata Bay', 'Nkhotakota', 'Northern', 'Nsanje', 'Ntcheu', 'Ntchisi', 'Phalombe', 'Rumphi', 'Salima', 'Southern', 'Thyolo', 'Zomba']


In [70]:
print(set(disease_df[(disease_df['country']=='Malawi')]['year']))
print(set(crop_df[(crop_df['country']=='Malawi')]['harvest_year']))

{2010, 2004, 2015}
{1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023}


- 그냥 sum해서 합친 적 있는 것 같은데(kenya) 그게 괜찮은지 다시 고민해보기 -> 면적 가중합
- malawi mapping

In [91]:
# 1. 말라위 size_map (면적: km²)
# (Key: 주(Region), Value: (Key: 지구(District), Value: 면적))
malawi_size_map = {
    'Northern Region': {
        'Chitipa': 4288.0,
        'Karonga': 3355.0,
        'Likoma': 18.0,  # 면적이 매우 작음
        'Mzimba': 10430.0,
        'Nkhata Bay': 4071.0,
        'Rumphi': 4769.0,
    },
    'Central Region': {
        'Dedza': 3624.0,
        'Dowa': 3041.0,
        'Kasungu': 7878.0,
        'Lilongwe': 6159.0,
        'Mchinji': 3356.0,
        'Nkhotakota': 4259.0,
        'Ntcheu': 3424.0,
        'Ntchisi': 1655.0,
        'Salima': 2196.0,
    },
    'Southern Region': {
        'Balaka': 2193.0,
        'Blantyre': 2012.0,
        'Chikwawa': 4755.0,
        'Chiradzulu': 767.0,
        'Machinga': 3771.0,
        'Mangochi': 6273.0,
        'Mulanje': 2056.0,
        'Mwanza': 850.0,
        'Neno': 1469.0,
        'Nsanje': 1942.0,
        'Phalombe': 1394.0,
        'Thyolo': 1715.0,
        'Zomba': 2580.0,
    }
}

In [92]:
# 2. 말라위 'district_to_region_map' (케냐의 county_to_province_map 역할)
# (Key: admin_1 원본, Value: standard_region)
malawi_district_to_region_map = {
    # Northern Region (6 지구)
    'Chitipa': 'Northern Region',
    'Karonga': 'Northern Region',
    'Likoma': 'Northern Region',
    'Mzimba': 'Northern Region',
    'Nkhata Bay': 'Northern Region',
    'Rumphi': 'Northern Region',

    # Central Region (9 지구)
    'Dedza': 'Central Region',
    'Dowa': 'Central Region',
    'Kasungu': 'Central Region',
    'Lilongwe': 'Central Region',
    'Mchinji': 'Central Region',
    'Nkhotakota': 'Central Region',
    'Ntcheu': 'Central Region',
    'Ntchisi': 'Central Region',
    'Salima': 'Central Region',

    # Southern Region (13 지구)
    'Balaka': 'Southern Region',
    'Blantyre': 'Southern Region',
    'Chikwawa': 'Southern Region',
    'Chiradzulu': 'Southern Region',
    'Machinga': 'Southern Region',
    'Mangochi': 'Southern Region',
    'Mulanje': 'Southern Region',
    'Mwanza': 'Southern Region',
    'Neno': 'Southern Region',
    'Nsanje': 'Southern Region',
    'Phalombe': 'Southern Region',
    'Thyolo': 'Southern Region',
    'Zomba': 'Southern Region',

    # 1-level (주) 이름도 매핑
    'Central': 'Central Region',
    'Northern': 'Northern Region',
    'Southern': 'Southern Region',
}

In [93]:
# (오류 방지)
try:
    # --- 1. (준비) 맵 정의 및 평탄화 ---
    # malawi_size_map, malawi_district_to_region_map은
    # 위 섹션에서 복사/붙여넣기 하시면 됩니다.

    # 'flat_size_map' 생성 (malawi_size_map으로부터)
    flat_size_map = {
        district: size
        for region_districts in malawi_size_map.values()
        for district, size in region_districts.items()
    }

    # --- 2. 말라위 데이터 분리 ---
    # 'Malawi'가 아닌 다른 나라 데이터는 그대로 보존
    df_others = crop_df[crop_df['country'] != 'Malawi'].copy()

    # 말라위 데이터만 추출
    df_malawi = crop_df[crop_df['country'] == 'Malawi'].copy()

    if df_malawi.empty:
        print("경고: crop_df에 'Malawi' 데이터가 없습니다. 작업을 건너뜁니다.")
        crop_df = df_others.copy() # 말라위가 없으면 df_others가 최종본
    else:
        # 'admin_1'을 매핑 사전을 이용해 'standard_region'으로 변환
        # 'flat_size_map'은 28개 지구만 포함
        df_malawi['size'] = df_malawi['admin_1'].map(flat_size_map)
        # 'admin_1'이 'Central' 등 주(Region) 이름일 경우 size는 NaN -> 0
        df_malawi['size'] = df_malawi['size'].fillna(0)

        # 'standard_region' 매핑 (31개 항목 모두 매핑)
        df_malawi['standard_region'] = df_malawi['admin_1'].map(malawi_district_to_region_map)

        unmapped = df_malawi[df_malawi['standard_region'].isnull()]['admin_1'].unique()
        if len(unmapped) > 0:
            print(f"경고: [standard_region] 매핑에 실패한 지역이 있습니다 -> {unmapped}")

        # [수정됨] 집계할 컬럼 리스트 (원본 crop_df 컬럼 기준)
        agg_columns = ['Cereals_Prod_Ton', 'Roots_tubers_Prod_Ton', 'Legumes_pulses_Prod_Ton', 'Cash_crops_Prod_Ton']

        # --- 3. 'total_production' (분자) 계산 ---
        total_prod_cols = []
        for col in agg_columns:
            prod_col_name = f'total_prod_{col}'
            df_malawi[prod_col_name] = df_malawi[col] * df_malawi['size']
            total_prod_cols.append(prod_col_name)

        # --- 4. Groupby 및 합산 ---
        group_keys = ['country','country_code', 'harvest_year', 'standard_region']
        cols_to_sum = total_prod_cols + ['size']

        # 매핑 실패한 행(standard_region=NaN)은 집계에서 제외
        df_malawi_grouped = df_malawi.dropna(subset=['standard_region'])

        if df_malawi_grouped.empty:
            print("오류: 말라위 데이터가 모두 매핑에 실패하여 집계할 수 없습니다.")
            df_malawi_agg = pd.DataFrame(columns=df_others.columns) # 빈 프레임
        else:
            df_malawi_agg = df_malawi_grouped.groupby(group_keys)[cols_to_sum].sum().reset_index()

            # --- 5. 최종 가중 평균 계산 ---
            for col in agg_columns:
                prod_col_name = f'total_prod_{col}'
                df_malawi_agg[col] = np.where(
                    df_malawi_agg['size'] == 0,
                    np.nan,  # 면적이 0이면 결과는 NaN
                    df_malawi_agg[prod_col_name] / df_malawi_agg['size']
                )
                df_malawi_agg = df_malawi_agg.drop(columns=[prod_col_name])

            # 'size' 열 삭제 (루프 바깥에서!)
            df_malawi_agg = df_malawi_agg.drop(columns=['size'])

            print(f"1단계: 'Malawi' 데이터 {len(df_malawi)}행을 {len(df_malawi_agg)}행으로 집계 완료.")

            # --- 6. 스키마 맞추기 ---
            df_malawi_agg = df_malawi_agg.rename(columns={'standard_region': 'admin_1'})
            df_malawi_agg['L1_GID'] = np.nan

        # --- 7. 다른 국가 데이터와 다시 합치기 ---
        crop_df = pd.concat([df_others, df_malawi_agg], ignore_index=True, sort=False)

        print(f"--- [최종 작업 완료] 말라위(Malawi) 데이터 정리 완료 ---")

        # --- 검증 ---
        print("\n--- 검증 (말라위 3개 주) ---")
        # 'standard region name'에서 'Area under ...' 제외
        standard_malawi_set = set(['Central Region', 'Northern Region', 'Southern Region'])

        final_malawi_regions = set(crop_df[crop_df['country'] == 'Malawi']['admin_1'].dropna())

        print(f"데이터에만 있는 지역 (set()여야 함): {final_malawi_regions - standard_malawi_set}")
        print(f"데이터에 누락된 지역 (set()여야 함): {standard_malawi_set - final_malawi_regions}")

except NameError as e:
    print(f"오류: '{e}' 변수가 정의되지 않았습니다. (crop_df, malawi_size_map 등 확인)")
except KeyError as e:
    print(f"오류: DataFrame에 '{e.key}' 컬럼이 없습니다.")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")

1단계: 'Malawi' 데이터 1265행을 141행으로 집계 완료.
--- [최종 작업 완료] 말라위(Malawi) 데이터 정리 완료 ---

--- 검증 (말라위 3개 주) ---
데이터에만 있는 지역 (set()여야 함): set()
데이터에 누락된 지역 (set()여야 함): set()


In [96]:
until_malawi_df = crop_df

##**Mali**

#### **mapping pair**
- 'Ségou':'Segou',
- 'Timbuktu':'Tombouctou'

#### **특이사항**

In [97]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Mali']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Mali']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Mali']['region_name'])))

crop region name
['Bamako', 'Gao', 'Kayes', 'Kidal', 'Koulikoro', 'Mopti', 'Segou', 'Sikasso', 'Ségou', 'Timbuktu', 'Tombouctou']

standard region name
['Bamako', 'Gao', 'Kayes', 'Kidal', 'Koulikoro', 'Mopti', 'Segou', 'Sikasso', 'Tombouctou']



mismatch list
['Ségou', 'Timbuktu']


In [99]:
print(set(disease_df[(disease_df['country']=='Mali')]['year']))
print(set(crop_df[(crop_df['country']=='Mali')]['harvest_year']))

{2001, 2012, 2006}
{1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022}


In [100]:
mali_mapping_dict = {
    'Ségou':'Segou',
    'Timbuktu':'Tombouctou'
}

crop_df.loc[crop_df['country'] == 'Mali', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Mali', 'admin_1'].replace(mali_mapping_dict)

print("disease data after mapping")
new_mali_df = crop_df[crop_df['country']=='Mali']
print(new_mali_df)

set(new_mali_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Mali']['ADM1_NAME'])

disease data after mapping
     country country_code     admin_1     L1_GID  harvest_year  \
1799    Mali           ML      Bamako  L1MLI1943          1974   
1800    Mali           ML      Bamako  L1MLI1943          1975   
1801    Mali           ML      Bamako  L1MLI1943          1976   
1802    Mali           ML      Bamako  L1MLI1943          1977   
1803    Mali           ML      Bamako  L1MLI1943          1978   
...      ...          ...         ...        ...           ...   
2293    Mali          MLI  Tombouctou  L1MLI1951          2010   
2294    Mali          MLI  Tombouctou  L1MLI1951          2012   
2295    Mali          MLI  Tombouctou  L1MLI1951          2013   
2296    Mali          MLI  Tombouctou  L1MLI1951          2014   
2297    Mali          MLI  Tombouctou  L1MLI1951          2018   

      Cereals_Prod_Ton  Roots_tubers_Prod_Ton  Legumes_pulses_Prod_Ton  \
1799          1.406445                    0.0                      0.0   
1800          1.088823          

set()

In [101]:
until_mali_df = crop_df

##**Mozambique, Namibia, Niger**

#### **mapping pair**
- mozambique
  - 'Nassa': 'Lago niassa'

#### **특이사항**
- namibia는 disease: {2013}, crop: {1995} 으로 시점이 맞지 않아 crop의 Namibia 데이터 삭제함
- niger도 disease: {2012, 2006}, crop: {1998} 으로 시점이 맞지 않아 crop의 niger 데이터 삭제함


In [104]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Mozambique']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Mozambique']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Mozambique']['region_name'])))

crop region name
['Cabo Delgado', 'Gaza', 'Inhambane', 'Manica', 'Maputo', 'Nampula', 'Nassa', 'Niassa', 'Sofala', 'Tete', 'Zambezia']

standard region name
['Cabo Delgado', 'Gaza', 'Inhambane', 'Lago niassa', 'Manica', 'Maputo', 'Nampula', 'Niassa', 'Sofala', 'Tete', 'Zambezia']



mismatch list
['Nassa']


In [109]:
print(set(disease_df[(disease_df['country']=='Niger')]['year']))
print(set(crop_df[(crop_df['country']=='Niger')]['harvest_year']))

{2012, 2006}
{1998}


In [106]:
mozambique_mapping_dict = {
    'Nassa': 'Lago niassa'
}

crop_df.loc[crop_df['country'] == 'Mozambique', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Mozambique', 'admin_1'].replace(mozambique_mapping_dict)

print("disease data after mapping")
new_mozambique_df = crop_df[crop_df['country']=='Mozambique']
print(new_mozambique_df)

set(new_mozambique_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Mozambique']['ADM1_NAME'])

disease data after mapping
         country country_code       admin_1     L1_GID  harvest_year  \
2298  Mozambique          MOZ  Cabo Delgado  L1MOZ2019          1999   
2299  Mozambique          MOZ  Cabo Delgado  L1MOZ2019          2012   
2300  Mozambique          MOZ  Cabo Delgado  L1MOZ2019          2014   
2301  Mozambique          MOZ  Cabo Delgado  L1MOZ2019          2015   
2302  Mozambique          MOZ          Gaza  L1MOZ2020          1999   
...          ...          ...           ...        ...           ...   
2546  Mozambique           MZ        Niassa  L1MOZ2026          2018   
2547  Mozambique           MZ        Niassa  L1MOZ2026          2019   
2548  Mozambique           MZ        Niassa  L1MOZ2026          2020   
2549  Mozambique           MZ        Niassa  L1MOZ2026          2021   
2550  Mozambique           MZ        Niassa  L1MOZ2026          2022   

      Cereals_Prod_Ton  Roots_tubers_Prod_Ton  Legumes_pulses_Prod_Ton  \
2298          0.000000            

set()

In [108]:
#Namibia 삭제
crop_df = crop_df[~(crop_df['country']=='Namibia')]

In [110]:
#Niger 삭제
crop_df = crop_df[~(crop_df['country']=='Niger')]

In [111]:
until_niger_df = crop_df

##**Rwanda**

#### **mapping pair**
- 'Amajyaruguru': 'North/Amajyaruguru',
- 'Amajyepfo': 'South/Amajyepfo',
- 'Iburasirazuba': 'East/Iburasirazuba',
- 'Iburengerazuba': 'West/Iburengerazuba',
- 'Umujyi wa Kigali': 'Kigali City/Umujyi wa Kigali'

#### **특이사항**
- disease: {2010, 2019, 2014}, crop: {2016, 2013, 2014, 2015} 이라서 disease에서 2014에 대해서만 crop data 존재

In [115]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Rwanda']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Rwanda']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Rwanda']['region_name'])))

crop region name
['Amajyaruguru', 'Amajyepfo', 'Iburasirazuba', 'Iburengerazuba', 'Umujyi wa Kigali']

standard region name
['East/Iburasirazuba', 'Kigali City/Umujyi wa Kigali', 'North/Amajyaruguru', 'South/Amajyepfo', 'West/Iburengerazuba']



mismatch list
['Amajyaruguru', 'Amajyepfo', 'Iburasirazuba', 'Iburengerazuba', 'Umujyi wa Kigali']


In [116]:
rwanda_mapping_dict = {
    'Amajyaruguru': 'North/Amajyaruguru',
    'Amajyepfo': 'South/Amajyepfo',
    'Iburasirazuba': 'East/Iburasirazuba',
    'Iburengerazuba': 'West/Iburengerazuba',
    'Umujyi wa Kigali': 'Kigali City/Umujyi wa Kigali'
}

crop_df.loc[crop_df['country'] == 'Rwanda', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Rwanda', 'admin_1'].replace(rwanda_mapping_dict)

print("disease data after mapping")
new_rwanda_df = crop_df[crop_df['country']=='Rwanda']
print(new_rwanda_df)

set(new_rwanda_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Rwanda']['ADM1_NAME'])

disease data after mapping
     country country_code                       admin_1     L1_GID  \
2566  Rwanda          RWA            North/Amajyaruguru  L1RWA2739   
2567  Rwanda          RWA            North/Amajyaruguru  L1RWA2739   
2568  Rwanda          RWA            North/Amajyaruguru  L1RWA2739   
2569  Rwanda          RWA            North/Amajyaruguru  L1RWA2739   
2570  Rwanda          RWA               South/Amajyepfo  L1RWA2740   
2571  Rwanda          RWA               South/Amajyepfo  L1RWA2740   
2572  Rwanda          RWA               South/Amajyepfo  L1RWA2740   
2573  Rwanda          RWA               South/Amajyepfo  L1RWA2740   
2574  Rwanda          RWA            East/Iburasirazuba  L1RWA2741   
2575  Rwanda          RWA            East/Iburasirazuba  L1RWA2741   
2576  Rwanda          RWA            East/Iburasirazuba  L1RWA2741   
2577  Rwanda          RWA            East/Iburasirazuba  L1RWA2741   
2578  Rwanda          RWA           West/Iburengerazuba  L1RWA2

set()

In [117]:
print(set(disease_df[(disease_df['country']=='Rwanda')]['year']))
print(set(crop_df[(crop_df['country']=='Rwanda')]['harvest_year']))

{2010, 2019, 2014}
{2016, 2013, 2014, 2015}


In [118]:
until_rwanda_df = crop_df

##**Senegal**

#### **mapping pair**
- 'Saint-Louis':'Saint louis',
- 'Sédhiou':'Sedhiou',
- 'Thiès':'Thies'

#### **특이사항**
- disease : {2017, 2010, 2005}
- crop : {2016, 2017, 2018, 2019, 1992, 1993, 1994, 1995, 1996, 1997, 2009, 2010, 2011, 2012, 2013, 2014, 2015}
- diseases, crop의 시점이 위와 같아서 disease에서 2005년은 crop 데이터가 없음

In [123]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Senegal']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Senegal']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Senegal']['region_name'])))

crop region name
['Dakar', 'Diourbel', 'Fatick', 'Kaffrine', 'Kaolack', 'Kolda', 'Louga', 'Saint-Louis', 'Sédhiou', 'Tambacounda', 'Thiès', 'Ziguinchor']

standard region name
['Dakar', 'Diourbel', 'Fatick', 'Kaffrine', 'Kaolack', 'Kedougou', 'Kolda', 'Louga', 'Matam', 'Saint louis', 'Sedhiou', 'Tambacounda', 'Thies', 'Ziguinchor']



mismatch list
['Saint-Louis', 'Sédhiou', 'Thiès']


In [124]:
senegal_mapping_dict = {
    'Saint-Louis':'Saint louis',
    'Sédhiou':'Sedhiou',
    'Thiès':'Thies'
}

crop_df.loc[crop_df['country'] == 'Senegal', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Senegal', 'admin_1'].replace(senegal_mapping_dict)

print("disease data after mapping")
new_senegal_df = crop_df[crop_df['country']=='Senegal']
print(new_senegal_df)

set(new_senegal_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Senegal']['ADM1_NAME'])

disease data after mapping
      country country_code     admin_1     L1_GID  harvest_year  \
2586  Senegal          SEN       Dakar  L1SEN2775          1992   
2587  Senegal          SEN       Dakar  L1SEN2775          1993   
2588  Senegal          SEN       Dakar  L1SEN2775          1994   
2589  Senegal          SEN       Dakar  L1SEN2775          1995   
2590  Senegal          SEN       Dakar  L1SEN2775          1996   
...       ...          ...         ...        ...           ...   
2715  Senegal          SEN  Ziguinchor  L1SEN2788          2015   
2716  Senegal          SEN  Ziguinchor  L1SEN2788          2016   
2717  Senegal          SEN  Ziguinchor  L1SEN2788          2017   
2718  Senegal          SEN  Ziguinchor  L1SEN2788          2018   
2719  Senegal          SEN  Ziguinchor  L1SEN2788          2019   

      Cereals_Prod_Ton  Roots_tubers_Prod_Ton  Legumes_pulses_Prod_Ton  \
2586          0.000000               0.000000                 0.649596   
2587          0.0000

set()

In [125]:
print(set(disease_df[(disease_df['country']=='Senegal')]['year']))
print(set(crop_df[(crop_df['country']=='Senegal')]['harvest_year']))

{2017, 2010, 2005}
{2016, 2017, 2018, 2019, 1992, 1993, 1994, 1995, 1996, 1997, 2009, 2010, 2011, 2012, 2013, 2014, 2015}


In [126]:
until_senegal_df = crop_df

##**South Africa**

#### **mapping pair**
- 'Kwazulu-Natal':'KwaZulu-Natal'

#### **특이사항**

In [130]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='South Africa']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='South Africa']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='South Africa']['region_name'])))

crop region name
['Eastern Cape', 'Free State', 'Gauteng', 'KwaZulu-Natal', 'Kwazulu-Natal', 'Limpopo', 'Mpumalanga', 'North West', 'Northern Cape', 'Western Cape']

standard region name
['Eastern Cape', 'Free State', 'Gauteng', 'KwaZulu-Natal', 'Limpopo', 'Mpumalanga', 'North West', 'Northern Cape', 'Western Cape']



mismatch list
['Kwazulu-Natal']


In [131]:
sa_mapping_dict = {
    'Kwazulu-Natal':'KwaZulu-Natal'
}

crop_df.loc[crop_df['country'] == 'South Africa', 'admin_1'] = crop_df.loc[crop_df['country'] == 'South Africa', 'admin_1'].replace(sa_mapping_dict)

print("disease data after mapping")
new_sa_df = crop_df[crop_df['country']=='South Africa']
print(new_sa_df)

set(new_sa_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='South Africa']['ADM1_NAME'])

disease data after mapping
           country country_code        admin_1     L1_GID  harvest_year  \
2723  South Africa           ZA   Eastern Cape  L1ZAF3629          1979   
2724  South Africa           ZA   Eastern Cape  L1ZAF3629          1980   
2725  South Africa           ZA   Eastern Cape  L1ZAF3629          1981   
2726  South Africa           ZA   Eastern Cape  L1ZAF3629          1982   
2727  South Africa           ZA   Eastern Cape  L1ZAF3629          1983   
...            ...          ...            ...        ...           ...   
3135  South Africa          ZAF     Mpumalanga  L1ZAF3634          1998   
3136  South Africa          ZAF     North West  L1ZAF3635          1991   
3137  South Africa          ZAF     North West  L1ZAF3635          1995   
3138  South Africa          ZAF  Northern Cape  L1ZAF3636          1995   
3139  South Africa          ZAF  KwaZulu-Natal  L1ZAF3632          1998   

      Cereals_Prod_Ton  Roots_tubers_Prod_Ton  Legumes_pulses_Prod_Ton  

set()

In [133]:
print(set(disease_df[(disease_df['country']=='South Africa')]['year']))
print(set(crop_df[(crop_df['country']=='South Africa')]['harvest_year']))

{2016}
{1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022}


In [132]:
until_sa_df = crop_df

##**Togo**

#### **mapping pair**

#### **특이사항**
- disease: {2013}
- crop: {1994}
- 시점이 맞지 않아서 crop에서 togo 삭제함

In [137]:
# crop 데이터에서의 region name
print('crop region name')
print(sorted(set(crop_df[crop_df['country']=='Togo']['admin_1'])))
# standard 데이터에서의 region name
print('\nstandard region name')
print(sorted(set(standard_df[standard_df['ADM0_NAME']=='Togo']['ADM1_NAME'])))


# crop과 standard에서 불일치하는 region name(disease 이름 기준)
print('\n\n\nmismatch list')
print(sorted(set(mismatch_df[mismatch_df['country']=='Togo']['region_name'])))

crop region name
['Centre', 'Kara', 'Maritime', 'Plateaux', 'Savanes']

standard region name
['Centrale', 'Kara', 'Maritime', 'Plateaux', 'Savanes']



mismatch list
['Centre']


In [138]:
togo_mapping_dict = {
    'Centre':'Centrale'
}

crop_df.loc[crop_df['country'] == 'Togo', 'admin_1'] = crop_df.loc[crop_df['country'] == 'Togo', 'admin_1'].replace(togo_mapping_dict)

print("disease data after mapping")
new_togo_df = crop_df[crop_df['country']=='Togo']
print(new_togo_df)

set(new_togo_df['admin_1']) - set(standard_df[standard_df['ADM0_NAME']=='Togo']['ADM1_NAME'])

disease data after mapping
     country country_code   admin_1     L1_GID  harvest_year  \
3160    Togo          TGO  Centrale  L1TGO3017          1994   
3161    Togo          TGO   Savanes  L1TGO3021          1994   
3162    Togo          TGO  Plateaux  L1TGO3020          1994   
3163    Togo          TGO      Kara  L1TGO3018          1994   
3164    Togo          TGO  Maritime  L1TGO3019          1994   

      Cereals_Prod_Ton  Roots_tubers_Prod_Ton  Legumes_pulses_Prod_Ton  \
3160          4.814292              18.122578                 0.935306   
3161          2.508130              22.337291                 0.984796   
3162          4.809554              22.877123                 1.037678   
3163          4.777144              14.857812                 0.594726   
3164          1.251905              21.135989                 0.943650   

      Cash_crops_Prod_Ton  
3160                  0.0  
3161                  0.0  
3162                  0.0  
3163                  0.0  
316

set()

In [139]:
print(set(disease_df[(disease_df['country']=='Togo')]['year']))
print(set(crop_df[(crop_df['country']=='Togo')]['harvest_year']))

{2013}
{1994}


In [140]:
crop_df = crop_df[~(crop_df['country']=='Togo')]

In [141]:
until_togo_df = crop_df

#**최종**

In [145]:
final_crop_df = crop_df

In [146]:
crop_df

,country,country_code,admin_1,L1_GID,harvest_year,Cereals_Prod_Ton,Roots_tubers_Prod_Ton,Legumes_pulses_Prod_Ton,Cash_crops_Prod_Ton
0,Angola,AO,Bengo,L1AGO0035,1997,0.610000,3.500000,0.000000,0.000000
1,Angola,AO,Bengo,L1AGO0035,1998,0.700000,6.500000,0.000000,0.000000
2,Angola,AO,Bengo,L1AGO0035,1999,0.899956,6.000000,0.000000,0.000000
3,Angola,AO,Bengo,L1AGO0035,2000,1.199960,9.000000,0.000000,0.000000
4,Angola,AO,Bengo,L1AGO0035,2001,0.600031,8.995530,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
3516,Malawi,MWI,Northern Region,NaN,2015,2.616512,7.521773,1.001776,0.594552
3517,Malawi,MWI,Southern Region,NaN,2015,1.984910,2.290430,0.950203,0.525923
3518,Malawi,MWI,Central Region,NaN,2016,2.183967,13.829757,1.197420,745.549325
3519,Malawi,MWI,Northern Region,NaN,2016,2.604531,7.862324,0.739671,968.051002


In [147]:
final_crop_df.to_csv('processed_crop.csv', index=False)